# (Optional) Lab 6: Add observability to your agents

**Optional take-home lab:** Complete this after Lab 1. The lab also reuses the AgentCore Memory resource from Lab 4; in a self-paced account, the setup deploys that resource if it is absent.

If you enabled observability in Labs 1 and 4, this lab reuses their managed Gateway and Memory activity. It also makes one fresh correlated request so every participant has a deterministic trace to follow. This advanced lab builds the operational evidence chain around managed telemetry delivery, OpenTelemetry span contracts, redaction assertions, correlation, and evaluation gates.

## What you will do

1. Enable AgentCore Gateway and Memory application logs and managed traces.
2. Run a Strands agent that makes one real Gateway tool call.
3. Write and retrieve one synthetic Lab 4 incident event and observe extraction and consolidation.
4. Inspect Strands `invoke_agent`, `chat`, and `execute_tool` spans plus application span contracts.
5. Validate W3C trace context, safe identity fields, and sensitive-content redaction.
6. Correlate Gateway and Memory metrics, traces, and logs in CloudWatch.
7. Turn the observed signals into evaluation and release gates.
8. Remove the telemetry resources and synthetic Memory data created by this notebook.

**Estimated time:** 20–25 minutes. Managed telemetry and Memory extraction can take several minutes to appear.


## Understand the observability signals

| Evidence | Produced automatically | Add for your application |
| --- | --- | --- |
| Agent and model | Strands `invoke_agent` and `chat` spans, model, tokens, latency, status | Route, contract version, evidence outcome, business result |
| Tools and Gateway | Strands `execute_tool` plus Gateway request, error, throttle, and latency signals | Safe tool labels and expected authorization outcome |
| Graph and Workflow | Agent/model/tool spans inside each real operation | Node, edge, interrupt, resume, checkpoint, stage, attempt, claim ID |
| Swarm | Agent/model/tool spans inside each real operation | Handoff, turn, limit, and evidence status |
| Memory | Real event, retrieval, extraction, and consolidation signals | Incident, operation, purpose, and result |
| A2A | Agent/model/tool spans inside each service | Trace/session context, peer, skill, retry, timeout, status |

Start with the operator's questions: metrics show whether the fleet changed, traces show which request and operation changed, and logs explain that component's details. Evaluations decide whether the observed behavior is acceptable for release.

AgentCore cannot infer graph nodes, workflow stages, swarm handoffs, or A2A skill names. The notebook adds illustrative spans for those concepts so you can inspect the attribute contract without rerunning Labs 2–5. Gateway and Memory operations are real.


## Setup

Run the `%pip` cell first. If it installs or upgrades anything, restart the kernel and choose **Run > Run All Below**.

Application spans stay in memory for immediate inspection. The real Gateway separately sends its managed logs and traces to CloudWatch.

In [ ]:
%pip install -r requirements.txt

In [ ]:
import json
import os
import re
import sys
import time
import uuid
from collections import defaultdict
from datetime import datetime, timezone
from pathlib import Path

# Keep GenAI span fields, but redact prompts, tool arguments, and tool results.
os.environ["OTEL_SEMCONV_STABILITY_OPT_IN"] = (
    "gen_ai_latest_experimental,gen_ai_span_attributes_only,gen_ai_unredacted_attributes="
)

import boto3
import httpx
from botocore.exceptions import ClientError
from bedrock_agentcore.memory import MemoryClient
from opentelemetry import propagate, trace
from opentelemetry.sdk.trace.export import SimpleSpanProcessor
from opentelemetry.sdk.trace.export.in_memory_span_exporter import InMemorySpanExporter
from strands import Agent
from strands.models import BedrockModel
from strands.telemetry import StrandsTelemetry
from strands.tools.mcp import MCPClient

sys.path.insert(0, str(Path("..").resolve()))
from utils.observability import (
    delete_observability_state,
    load_observability_state,
    save_observability_state,
)
from utils.stacks import cleanup, find_or_deploy
from utils.validation import write_validation_artifact

span_exporter = InMemorySpanExporter()
telemetry = StrandsTelemetry()
telemetry.tracer_provider.add_span_processor(SimpleSpanProcessor(span_exporter))
tracer = trace.get_tracer("opn307.observability")
session_id = f"opn307-observability-{uuid.uuid4().hex}"

print("Session ID:", session_id)
print("Sensitive prompt and tool content: redacted")


## Establish the managed Gateway and Identity boundary

The next cell finds the Lab 1 Gateway and gets a one-hour Cognito client-credentials token. The token and AWS account ID are never recorded in telemetry. The root span records only the authentication mode and a non-identifying caller type.


In [ ]:
region = boto3.Session().region_name
observability_state_path = Path("..") / ".lab6-observability-state.json"
gateway_outputs = find_or_deploy("agents-as-tools", "../01-agents-as-tools/infra/agents-as-tools.yaml")
memory_outputs = find_or_deploy("swarm", "../04-swarm/infra/swarm.yaml")
outputs = gateway_outputs

identity = boto3.client("sts", region_name=region).get_caller_identity()
account_id = identity["Account"]
partition = identity["Arn"].split(":", 2)[1]
gateway_id = outputs.get("GatewayId") or outputs["GatewayUrl"].split("//", 1)[1].split(".", 1)[0]
gateway_arn = outputs.get("GatewayArn") or f"arn:{partition}:bedrock-agentcore:{region}:{account_id}:gateway/{gateway_id}"
gateway_log_group = f"/aws/vendedlogs/bedrock-agentcore/gateway/APPLICATION_LOGS/{gateway_id}"

memory_id = memory_outputs["MemoryId"]
memory_arn = memory_outputs.get("MemoryArn") or f"arn:{partition}:bedrock-agentcore:{region}:{account_id}:memory/{memory_id}"
memory_log_group = f"/aws/vendedlogs/bedrock-agentcore/memory/APPLICATION_LOGS/{memory_id}"

cognito = boto3.client("cognito-idp", region_name=region)
client_secret = cognito.describe_user_pool_client(
    UserPoolId=outputs["UserPoolId"], ClientId=outputs["ClientId"]
)["UserPoolClient"]["ClientSecret"]
token_response = httpx.post(
    outputs["TokenUrl"],
    data={"grant_type": "client_credentials", "scope": outputs["Scope"]},
    auth=(outputs["ClientId"], client_secret),
)
token_response.raise_for_status()
headers = {"Authorization": f"Bearer {token_response.json()['access_token']}"}

print("Gateway ID      :", gateway_id)
print("Gateway target  :", outputs["TargetName"])
print("Auth mode       : Cognito client credentials")
print("Gateway log group:", gateway_log_group)
print("Memory ID       :", memory_id)
print("Memory log group:", memory_log_group)


## Enable managed service telemetry

This idempotent setup first loads any telemetry configuration created by the optional observability cells in Labs 1 and 4. It then enables anything still missing: CloudWatch Transaction Search, 100% trace indexing, Gateway `APPLICATION_LOGS` and `TRACES` delivery, and Memory `APPLICATION_LOGS` delivery. The shared ownership record lets the final cleanup cell restore or remove only workshop-owned changes.

Transaction Search is an account-wide asynchronous setting. If trace delivery is still activating, the cell enables the available telemetry and tells you to rerun the same cell in a few minutes. The cell is safe to rerun.


In [ ]:
logs = boto3.client("logs", region_name=region)
xray = boto3.client("xray", region_name=region)


def clean_name(value):
    return re.sub(r"[^\w-]", "-", value)[:60]


def is_missing(error):
    return error.response["Error"]["Code"] in {
        "ResourceNotFoundException", "NotFoundException", "404"
    }


def get_source(name):
    try:
        return logs.get_delivery_source(name=name)["deliverySource"]
    except ClientError as error:
        if is_missing(error):
            return None
        raise


def get_destination(name):
    try:
        return logs.get_delivery_destination(name=name)["deliveryDestination"]
    except ClientError as error:
        if is_missing(error):
            return None
        raise


def find_delivery(source_name, destination_arn):
    for page in logs.get_paginator("describe_deliveries").paginate():
        for delivery in page.get("deliveries", []):
            if (
                delivery["deliverySourceName"] == source_name
                and delivery["deliveryDestinationArn"] == destination_arn
            ):
                return delivery
    return None


def put_source(name, log_type, resource_arn):
    existing = get_source(name)
    if existing:
        return existing["arn"], False
    source = logs.put_delivery_source(
        name=name,
        logType=log_type,
        resourceArn=resource_arn,
    )["deliverySource"]
    return source["arn"], True


def put_destination(name, destination_type, log_group_arn=None):
    existing = get_destination(name)
    if existing:
        return existing["arn"], False
    kwargs = {
        "name": name,
        "deliveryDestinationType": destination_type,
    }
    if log_group_arn:
        kwargs["deliveryDestinationConfiguration"] = {"destinationResourceArn": log_group_arn}
        kwargs["outputFormat"] = "json"
    destination = logs.put_delivery_destination(**kwargs)["deliveryDestination"]
    return destination["arn"], True


def create_delivery(source_name, destination_arn):
    existing = find_delivery(source_name, destination_arn)
    if existing:
        return existing["id"], False
    delivery = logs.create_delivery(
        deliverySourceName=source_name,
        deliveryDestinationArn=destination_arn,
    )["delivery"]
    return delivery["id"], True


def wait_for_trace_destination(expected, timeout_seconds=300, required=True):
    deadline = time.monotonic() + timeout_seconds
    while True:
        state = xray.get_trace_segment_destination()
        if state.get("Destination") == expected and state.get("Status") != "PENDING":
            return state
        if time.monotonic() >= deadline:
            if required:
                raise TimeoutError(
                    f"Trace destination did not become ready: expected {expected}, got {state}"
                )
            print(
                f"Trace destination {expected} is still {state.get('Status')}; "
                "AWS will complete the accepted update asynchronously."
            )
            return state
        time.sleep(5)


current_trace_state = xray.get_trace_segment_destination()
current_trace_destination = current_trace_state.get("Destination") or "XRay"
transaction_search_state = current_trace_state
if current_trace_state.get("Status") == "PENDING":
    transaction_search_state = wait_for_trace_destination(
        current_trace_destination, timeout_seconds=60, required=False
    )
if "observability_cleanup" not in globals():
    observability_cleanup = load_observability_state(
        observability_state_path,
        region=region,
    )
    if observability_cleanup.get("gateway_id") == gateway_id:
        print("Reusing Gateway telemetry enabled in Lab 1.")
    if observability_cleanup.get("memory_id") == memory_id:
        print("Reusing Memory telemetry enabled in Lab 4.")
if observability_cleanup.get("previous_trace_destination") is None:
    observability_cleanup["previous_trace_destination"] = current_trace_destination
observability_cleanup.setdefault("transaction_search_changed", False)
observability_cleanup.setdefault("resource_policy_created", False)
observability_cleanup.setdefault("previous_resource_policy_document", None)
observability_cleanup.setdefault("previous_indexing_percentage", None)
observability_cleanup.setdefault("indexing_changed", False)
observability_cleanup.setdefault("log_groups", {})
observability_cleanup.setdefault("memory_event", None)
observability_cleanup.setdefault("memory_records", [])

resource_policy_name = "AgentCoreTransactionSearch"
existing_policies = logs.describe_resource_policies(limit=50).get("resourcePolicies", [])
existing_policy = next(
    (item for item in existing_policies if item.get("policyName") == resource_policy_name),
    None,
)
policy_existed = existing_policy is not None

if current_trace_destination != "CloudWatchLogs":
    if policy_existed and observability_cleanup["previous_resource_policy_document"] is None:
        observability_cleanup["previous_resource_policy_document"] = existing_policy.get("policyDocument")
    policy = {
        "Version": "2012-10-17",
        "Statement": [{
            "Sid": "TransactionSearchXRayAccess", "Effect": "Allow",
            "Principal": {"Service": "xray.amazonaws.com"}, "Action": "logs:PutLogEvents",
            "Resource": [
                f"arn:{partition}:logs:{region}:{account_id}:log-group:aws/spans:*",
                f"arn:{partition}:logs:{region}:{account_id}:log-group:/aws/application-signals/data:*",
            ],
            "Condition": {
                "ArnLike": {"aws:SourceArn": f"arn:{partition}:xray:{region}:{account_id}:*"},
                "StringEquals": {"aws:SourceAccount": account_id},
            },
        }],
    }
    logs.put_resource_policy(policyName=resource_policy_name, policyDocument=json.dumps(policy))
    xray.update_trace_segment_destination(Destination="CloudWatchLogs")
    observability_cleanup["transaction_search_changed"] = True
    observability_cleanup["resource_policy_created"] |= not policy_existed
    transaction_search_state = wait_for_trace_destination(
        "CloudWatchLogs", timeout_seconds=60, required=False
    )

# Index every trace created during the lab so a single Gateway invocation is deterministic.
# The cleanup cell restores the account's previous percentage.
indexing_rules = xray.get_indexing_rules().get("IndexingRules", [])
default_indexing_rule = next(
    (item for item in indexing_rules if item.get("Name") == "Default"),
    None,
)
current_indexing_percentage = (
    default_indexing_rule.get("Rule", {}).get("Probabilistic", {}).get("DesiredSamplingPercentage", 1.0)
    if default_indexing_rule else 1.0
)
if current_indexing_percentage != 100.0:
    if observability_cleanup["previous_indexing_percentage"] is None:
        observability_cleanup["previous_indexing_percentage"] = current_indexing_percentage
    xray.update_indexing_rule(
        Name="Default",
        Rule={"Probabilistic": {"DesiredSamplingPercentage": 100.0}},
    )
    observability_cleanup["indexing_changed"] = True


def ensure_log_group(name):
    exists = any(
        item.get("logGroupName") == name
        for item in logs.describe_log_groups(logGroupNamePrefix=name, limit=50).get("logGroups", [])
    )
    if not exists:
        logs.create_log_group(logGroupName=name)
    logs.put_retention_policy(logGroupName=name, retentionInDays=7)
    observability_cleanup["log_groups"][name] = (
        observability_cleanup["log_groups"].get(name, False) or not exists
    )


ensure_log_group(gateway_log_group)
ensure_log_group(memory_log_group)

slug = clean_name(gateway_id)
application_source = clean_name(f"maw-{slug}-application-source")
application_destination = clean_name(f"maw-{slug}-application-destination")
trace_source = clean_name(f"maw-{slug}-trace-source")
trace_destination_name = clean_name(f"maw-{slug}-trace-destination")

_, created = put_source(application_source, "APPLICATION_LOGS", gateway_arn)
observability_cleanup["delivery_sources"][application_source] = (
    observability_cleanup["delivery_sources"].get(application_source, False) or created
)
app_destination_arn, created = put_destination(
    application_destination,
    "CWL",
    f"arn:{partition}:logs:{region}:{account_id}:log-group:{gateway_log_group}",
)
observability_cleanup["delivery_destinations"][application_destination] = (
    observability_cleanup["delivery_destinations"].get(application_destination, False) or created
)
delivery_id, created = create_delivery(application_source, app_destination_arn)
if created:
    observability_cleanup["deliveries"]["application"] = delivery_id

_, created = put_source(trace_source, "TRACES", gateway_arn)
observability_cleanup["delivery_sources"][trace_source] = (
    observability_cleanup["delivery_sources"].get(trace_source, False) or created
)
trace_destination_arn, created = put_destination(trace_destination_name, "XRAY")
observability_cleanup["delivery_destinations"][trace_destination_name] = (
    observability_cleanup["delivery_destinations"].get(trace_destination_name, False) or created
)

# X-Ray can accept the account-wide change before it is ready for a trace delivery.
# Keep the lab usable during that asynchronous window; rerunning this idempotent
# cell creates the trace delivery as soon as AWS reports the destination ready.
transaction_search_state = xray.get_trace_segment_destination()
trace_delivery_ready = (
    transaction_search_state.get("Destination") == "CloudWatchLogs"
    and transaction_search_state.get("Status") != "PENDING"
)
if trace_delivery_ready:
    delivery_id, created = create_delivery(trace_source, trace_destination_arn)
    if created:
        observability_cleanup["deliveries"]["trace"] = delivery_id


# Memory metrics are emitted by the service. Enable application logs and managed
# traces before inspecting extraction, consolidation, and API operations.
memory_slug = clean_name(memory_id)
memory_application_source = clean_name(f"maw-{memory_slug}-application-source")
memory_application_destination = clean_name(f"maw-{memory_slug}-application-destination")
memory_trace_source = clean_name(f"maw-{memory_slug}-trace-source")
memory_trace_destination_name = clean_name(f"maw-{memory_slug}-trace-destination")
_, created = put_source(memory_application_source, "APPLICATION_LOGS", memory_arn)
observability_cleanup["delivery_sources"][memory_application_source] = (
    observability_cleanup["delivery_sources"].get(memory_application_source, False) or created
)
memory_destination_arn, created = put_destination(
    memory_application_destination,
    "CWL",
    f"arn:{partition}:logs:{region}:{account_id}:log-group:{memory_log_group}",
)
observability_cleanup["delivery_destinations"][memory_application_destination] = (
    observability_cleanup["delivery_destinations"].get(memory_application_destination, False) or created
)
delivery_id, created = create_delivery(memory_application_source, memory_destination_arn)
if created:
    observability_cleanup["deliveries"]["memory-application"] = delivery_id

_, created = put_source(memory_trace_source, "TRACES", memory_arn)
observability_cleanup["delivery_sources"][memory_trace_source] = (
    observability_cleanup["delivery_sources"].get(memory_trace_source, False) or created
)
memory_trace_destination_arn, created = put_destination(
    memory_trace_destination_name,
    "XRAY",
)
observability_cleanup["delivery_destinations"][memory_trace_destination_name] = (
    observability_cleanup["delivery_destinations"].get(memory_trace_destination_name, False) or created
)
if trace_delivery_ready:
    delivery_id, created = create_delivery(memory_trace_source, memory_trace_destination_arn)
    if created:
        observability_cleanup["deliveries"]["memory-trace"] = delivery_id

if transaction_search_state.get("Status") == "PENDING":
    print("Transaction Search setup accepted; managed trace delivery is still activating. Rerun this setup cell in a few minutes.")
else:
    print("Transaction Search: enabled")
print("Trace indexing   : 100% for this lab")
print("Gateway app logs : enabled")
print("Memory app logs  : enabled")
if trace_delivery_ready:
    print("Gateway traces   : enabled")
    print("Memory traces    : enabled")
else:
    print("Gateway traces   : pending; rerun this cell in a few minutes")
    print("Memory traces    : pending; rerun this cell in a few minutes")
save_observability_state(observability_state_path, observability_cleanup)

The setup checkpoint confirms that the managed telemetry paths are ready:

![Notebook checkpoint confirming that Transaction Search, Gateway and Memory logs and traces are enabled.](images/01-telemetry-enabled.jpg)


## Build one application trace

The cell runs a real Strands agent and Gateway tool call, producing automatic `invoke_agent`, `chat`, and `execute_tool` spans. Inside that same local trace, it adds illustrative spans for Graph nodes, Workflow stages, Swarm handoffs, and an A2A peer boundary. Labs 2–5 contain the real pattern implementations.

The A2A example injects and extracts W3C trace context inside this notebook process. It demonstrates the instrumentation calls; it does not verify propagation across a network or service boundary.


In [ ]:
region = boto3.Session().region_name
nova = BedrockModel(model_id="us.amazon.nova-pro-v1:0", region_name=region)
sonnet = BedrockModel(model_id="global.anthropic.claude-sonnet-5", region_name=region)
model = sonnet

receipt_tools = MCPClient(
    url=outputs["GatewayUrl"], headers=headers,
    tool_filters={"allowed": [f"{outputs['TargetName']}___get_receipt"]},
)
observed_agent = Agent(
    name="receipt_observer", model=model,
    system_prompt=(
        "Use the receipt tool for the requested expense. Return merchant, date, total, and employee ID "
        "in at most two sentences."
    ),
    tools=[receipt_tools], callback_handler=None,
    trace_attributes={
        "session.id": session_id,
        "workshop.pattern": "agents-as-tools",
        "workshop.route": "receipt_observer",
        "workshop.contract.version": "1",
    },
)

agent_prompt = "Look up receipt expense-1042 and report its facts."
invoked_at = datetime.now(timezone.utc)
with tracer.start_as_current_span("expense-review-session", attributes={
    "session.id": session_id,
    "workshop.claim.id": "expense-1042",
    "workshop.identity.auth_mode": "cognito-client-credentials",
    "workshop.identity.caller_type": "machine-workload",
}) as session_span:
    agent_result = observed_agent(agent_prompt)

    # Illustrative pattern spans: these record the attribute contract but do not execute Labs 2–5.
    with tracer.start_as_current_span("graph.node manager_approval", attributes={
        "workshop.example": "illustrative", "workshop.pattern": "graph",
        "graph.version": "1", "graph.node": "manager_approval",
        "graph.decision": "required", "graph.interrupt.id": "approval-expense-1042",
        "graph.status": "interrupted",
    }) as span:
        span.add_event("graph.checkpoint", {"checkpoint.saved": True})

    with tracer.start_as_current_span("workflow.stage policy_review", attributes={
        "workshop.example": "illustrative", "workshop.pattern": "workflow",
        "workflow.claim.id": "expense-1042", "workflow.stage": "policy_review",
        "workflow.attempt": 1, "workflow.status": "completed",
    }):
        pass

    with tracer.start_as_current_span("swarm.handoff logs_to_code", attributes={
        "workshop.example": "illustrative", "workshop.pattern": "swarm",
        "swarm.incident.id": "incident-expense-1042",
        "swarm.source_agent": "logs_investigator", "swarm.destination_agent": "code_investigator",
        "swarm.turn": 2, "swarm.remaining_handoffs": 4, "swarm.evidence.status": "partial",
    }):
        pass

    # Local instrumentation example: inject, extract, then start a peer span in the same process.
    carrier = {}
    propagate.inject(carrier)
    peer_context = propagate.extract(carrier)
    with tracer.start_as_current_span(
        "a2a.peer policy_agent", context=peer_context, kind=trace.SpanKind.CLIENT,
        attributes={
            "workshop.example": "illustrative", "workshop.pattern": "a2a",
            "a2a.peer": "policy_agent", "a2a.skill": "expense-policy",
            "a2a.agent_card.version": "1", "a2a.remote.status": "simulated-local",
            "a2a.retry": 0,
        },
    ) as peer_span:
        peer_trace_id = format(peer_span.get_span_context().trace_id, "032x")
    session_trace_id = format(session_span.get_span_context().trace_id, "032x")

tool_results = [
    block["toolResult"]["content"]
    for message in observed_agent.messages for block in message["content"] if "toolResult" in block
]
tool_evidence = json.dumps(tool_results)
assert "expense-1042" in tool_evidence and "Cedar Cafe" in tool_evidence
assert peer_trace_id == session_trace_id

print("Invoked at        :", invoked_at.isoformat())
print("Application trace :", session_trace_id)
print("Local A2A example :", peer_trace_id)
print("Injected context  :", carrier.get("traceparent", "not exported"))
print("Gateway evidence  : expense-1042 / Cedar Cafe")
print("Agent answer      :", str(agent_result))


## Generate real AgentCore Memory telemetry

The next cell reuses Lab 4's incident-memory scenario. It writes a synthetic resolved `user-registration-service` Lambda incident to the Lab 4 Memory, waits for the semantic strategy to extract a long-term record, and retrieves it as a past incident. The application spans wrap real `CreateEvent` and `RetrieveMemoryRecords` calls and record only operation metadata, never the event text.

The event, any extracted records, and the Memory telemetry delivery created by this notebook are tracked for cleanup.


In [ ]:
memory = MemoryClient(region_name=region)
memory_data = boto3.client("bedrock-agentcore", region_name=region)
memory_actor_id = f"lab6-{session_id[-12:]}"
memory_session_id = f"{session_id}-memory"
memory_namespace = f"/incidents/{memory_actor_id}/"

memory_event_text = (
    "Lab 4 incident INC-0412: user-registration-service failed with AttributeError: "
    "'NoneType' object has no attribute 'lower' because the email value was missing. "
    "Resolution: validate required fields before string operations and backfill the missing email."
)
with tracer.start_as_current_span("application.memory CreateEvent", attributes={
    "session.id": session_id,
    "memory.operation": "CreateEvent",
    "memory.purpose": "incident-recall",
    "memory.actor.type": "on-call-team",
}) as create_span:
    existing_events = memory.list_events(
        memory_id=memory_id,
        actor_id=memory_actor_id,
        session_id=memory_session_id,
        max_results=2,
    )
    if existing_events:
        memory_event = existing_events[0]
        create_status = "already-exists"
    else:
        memory_event = memory.create_event(
            memory_id=memory_id,
            actor_id=memory_actor_id,
            session_id=memory_session_id,
            messages=[(memory_event_text, "USER")],
        )
        create_status = "accepted"
    create_span.set_attribute("memory.result.status", create_status)
    memory_trace_id = format(create_span.get_span_context().trace_id, "032x")

observability_cleanup["memory_event"] = {
    "memoryId": memory_id,
    "actorId": memory_actor_id,
    "sessionId": memory_session_id,
    "eventId": memory_event["eventId"],
}

new_records = []
for attempt in range(18):
    summaries = memory.list_memory_records(
        memoryId=memory_id,
        namespace=memory_namespace,
        maxResults=100,
    ).get("memoryRecordSummaries", [])
    new_records = summaries
    if new_records:
        break
    if attempt < 17:
        time.sleep(10)

with tracer.start_as_current_span("application.memory RetrieveMemoryRecords", attributes={
    "session.id": session_id,
    "memory.operation": "RetrieveMemoryRecords",
    "memory.purpose": "incident-recall",
    "memory.actor.type": "on-call-team",
}) as retrieve_span:
    retrieved_memories = memory.retrieve_memories(
        memory_id=memory_id,
        namespace=memory_namespace,
        query="INC-0412 user-registration-service missing email resolution",
        top_k=5,
    )
    retrieve_span.set_attribute("memory.result.count", len(retrieved_memories))
    retrieve_span.set_attribute(
        "memory.result.status", "complete" if new_records else "extraction-pending"
    )

observability_cleanup["memory_records"] = [
    {"memoryId": memory_id, "memoryRecordId": item["memoryRecordId"], "namespace": memory_namespace}
    for item in new_records
]

print("Memory event      :", memory_event["eventId"])
print("Memory session    :", memory_session_id)
print("Memory trace      :", memory_trace_id)
print("Extracted records :", len(new_records))
print("Retrieved records :", len(retrieved_memories))
print("Memory log group  :", memory_log_group)


## Inspect and validate the span tree

Look for the real Strands `invoke_agent`, `chat`, and `execute_tool` spans; the real Memory `CreateEvent` and `RetrieveMemoryRecords` spans; and the illustrative Graph, Workflow, Swarm, and A2A spans. Gateway and Memory use separate local traces but share one safe session identifier.

The next cell also asserts that the exact prompt, tool result, answer text, Memory event text, authorization token, and AWS account ID do not appear in any exported span attribute.


In [ ]:
finished_spans = list(span_exporter.get_finished_spans())
spans_by_id = {span.context.span_id: span for span in finished_spans}
children = defaultdict(list)
roots = []
for span in finished_spans:
    parent_id = span.parent.span_id if span.parent else None
    (children[parent_id] if parent_id in spans_by_id else roots).append(span)


def walk(span, depth=0):
    trace_id = format(span.context.trace_id, "032x")
    duration_ms = (span.end_time - span.start_time) / 1_000_000
    print(f"{'  ' * depth}{span.name:<48} {duration_ms:>8.1f} ms  {trace_id}")
    for child in sorted(children[span.context.span_id], key=lambda item: item.start_time):
        walk(child, depth + 1)


for root_span in sorted(roots, key=lambda item: item.start_time):
    walk(root_span)

names = [span.name for span in finished_spans]
required_span_checks = {
    "Strands agent": any(name.startswith("invoke_agent ") for name in names),
    "Strands model": "chat" in names,
    "Strands tool": any(name.startswith("execute_tool ") for name in names),
    "Illustrative Graph node": any(name.startswith("graph.node ") for name in names),
    "Illustrative Workflow stage": any(name.startswith("workflow.stage ") for name in names),
    "Illustrative Swarm handoff": any(name.startswith("swarm.handoff ") for name in names),
    "AgentCore Memory create": "application.memory CreateEvent" in names,
    "AgentCore Memory retrieve": "application.memory RetrieveMemoryRecords" in names,
    "Illustrative A2A peer": any(name.startswith("a2a.peer ") for name in names),
}
assert all(required_span_checks.values()), required_span_checks

attribute_blob = json.dumps(
    [{str(key): str(value) for key, value in span.attributes.items()} for span in finished_spans],
    sort_keys=True,
)
redaction_checks = {
    "prompt text absent": agent_prompt not in attribute_blob,
    "completion/tool text absent": "Cedar Cafe" not in attribute_blob,
    "serialized tool result absent": tool_evidence not in attribute_blob,
    "memory event text absent": memory_event_text not in attribute_blob,
    "authorization token absent": headers["Authorization"] not in attribute_blob,
    "account ID absent": account_id not in attribute_blob,
}
assert all(redaction_checks.values()), redaction_checks

print()
print("Required span contract:", required_span_checks)
print("Redaction checks       :", redaction_checks)


![Notebook output showing the real Strands agent, model, and Gateway tool-call span tree.](images/02-agent-span-tree.jpg)

## Define observability signals and evaluation gates

Use one evidence chain for operations and release decisions: a metric finds a fleet-level change, a trace isolates the request and operation, a linked log explains the component, and an evaluation decides whether the behavior is acceptable.

A trace answers one-request questions. Metrics and evaluations answer fleet and release questions.

- **Gateway:** `Invocations`, `Throttles`, `UserErrors`, `SystemErrors`, `Latency`, `Duration`, `TargetExecutionTime`, `OverheadLatency`
- **Runtime:** `Invocations`, `Latency`, errors, throttles, sessions, CPU, and memory. Runtime is not deployed in this workshop, so this is a deployment checklist.
- **Memory:** latency, errors, throttles, creation count, `CreateEvent` and `RetrieveMemoryRecords` operations, and extraction/consolidation logs
- **Evaluations:** routing, evidence transfer, graph outcomes, workflow accuracy, swarm evidence completeness, A2A contracts, final quality, and safety

The next check proves the local traces contain the fields those evaluations and investigations need. Gateway and Memory operations are real; the Graph, Workflow, Swarm-handoff, and A2A fields remain illustrative until you add the same instrumentation to the real Labs 2–5 implementations.


In [ ]:
def seen(attribute):
    return any(attribute in span.attributes for span in finished_spans)


production_contract = {
    "session": seen("session.id"),
    "route": seen("workshop.route"),
    "graph node/status": seen("graph.node") and seen("graph.status"),
    "workflow stage/attempt": seen("workflow.stage") and seen("workflow.attempt"),
    "swarm handoff/limit": seen("swarm.destination_agent") and seen("swarm.remaining_handoffs"),
    "memory create/retrieve": seen("memory.purpose") and seen("memory.operation"),
    "A2A peer/skill": seen("a2a.peer") and seen("a2a.skill"),
    "safe identity context": seen("workshop.identity.auth_mode") and seen("workshop.identity.caller_type"),
    "sensitive content redacted": all(redaction_checks.values()),
}
assert all(production_contract.values()), production_contract
for item, passed in production_contract.items():
    print(f"{'PASS' if passed else 'FAIL':4}  {item}")


## Correlate Gateway metrics, traces, and logs

If you enabled observability in Lab 1, CloudWatch already contains its Gateway requests. The fresh request in this notebook provides a known timestamp and session for comparison. The next cell waits for that recent Gateway application log; delivery is asynchronous, so continue if it has not arrived yet.

In [ ]:
events = []
for attempt in range(7):
    response = logs.filter_log_events(
        logGroupName=gateway_log_group,
        startTime=int((invoked_at.timestamp() - 30) * 1000),
        limit=20,
    )
    events = response.get("events", [])
    if events:
        break
    if attempt < 6:
        time.sleep(10)

print("Application-log events:", len(events))
print(events[-1]["message"][:800] if events else "No event yet. Refresh CloudWatch after a minute.")

1. Open **CloudWatch** in the notebook Region.
2. Choose **GenAI Observability** → **Bedrock AgentCore** → **Gateways** → the printed Gateway ID.
3. Set the time range to include the printed invocation time. Start with invocation, latency, error, and throttle signals.

![CloudWatch GenAI Observability showing eight Gateway invocations with no errors or throttles.](images/03-gateway-health.jpg)

4. If the managed trace is still being indexed, open **Logs Insights**, select the printed Gateway log group, and run:

```text
fields @timestamp, body.log as event, traceId, spanId
| sort @timestamp asc
| limit 20
```

The request lifecycle should repeat one `traceId` and `spanId`:

![CloudWatch Logs Insights showing the Gateway request lifecycle with a shared trace ID and span ID.](images/04-log-correlation.jpg)

5. When the managed trace appears, open it and select `AgentCore.Gateway.InvokeTool`.
6. Compare `execute_tool_latency_ms` with `overhead_latency_ms`, inspect status/error fields, and use trace/span IDs to join the trace to logs.

![Sanitized Gateway InvokeTool span metadata showing target execution and Gateway overhead latency.](images/07-gateway-trace-detail.jpg)

When the setup cell reports `Gateway traces: enabled`, the managed Gateway logs and traces reach CloudWatch. Trace indexing can lag behind metrics and log delivery. If the setup cell reports `pending`, rerun that idempotent cell after AWS finishes the Transaction Search update, invoke the tool again, and then use the views above. The Strands and illustrative application spans stay in the notebook's in-memory exporter in this lab. In production, export those OpenTelemetry spans through ADOT/OTLP and propagate trace context across every real HTTP or messaging boundary.


## Inspect Memory extraction and consolidation

If you enabled observability in Lab 4, its Memory metrics, managed traces, and application logs are already available. Compare that activity with the fresh operation in this notebook. A Memory call returns before the asynchronous strategy finishes its service-side work, so use all three signals to verify the lifecycle.

1. Open **Amazon Bedrock AgentCore** in the notebook Region.
2. Choose **Memory**, then the Memory ID printed by the notebook.
3. Compare short-term **Create events** and **Retrieve extracted memory** invocations, latency, and errors.
4. Open the long-term strategy and confirm **New memory extracted** increased after this run.

![AgentCore Memory showing real event creation, retrieval, and long-term extraction metrics.](images/05-memory-observability.jpg)

5. In **CloudWatch → Transaction Search**, set the time range to include Lab 4 and Lab 6, search for the Memory ID, and open a Memory service span to compare its operation, status, and duration.

6. In **CloudWatch → Log Analytics**, query the printed Memory log group. Project only safe lifecycle fields; do not display `body` because it can contain the event text.

```text
fields @timestamp, body.log as step, memory_strategy_id, namespace, actor_id, session_id
| filter session_id = "<memory session ID printed by the notebook>"
| sort @timestamp asc
| limit 20
```

The rows should progress through extraction and consolidation: processing input, starting the semantic strategy, extracting a memory, retrieving related records, choosing an add/update/no-op operation, and completing the write.

![CloudWatch Log Analytics showing the real Memory extraction and consolidation lifecycle.](images/06-memory-log-lifecycle.jpg)

The managed Memory span identifies one service operation and its duration. The Memory metrics show fleet health, and the lifecycle log shows what the asynchronous strategy did after the API returned.

In [ ]:
write_validation_artifact("observability", {
    "session_id": session_id,
    "application_trace_id": session_trace_id,
    "memory_trace_id": memory_trace_id,
    "memory_event_id": memory_event["eventId"],
    "memory_records_extracted": len(new_records),
    "memory_records_retrieved": len(retrieved_memories),
    "gateway_id": gateway_id,
    "gateway_tool_evidence": "expense-1042 / Cedar Cafe",
    "application_log_seen": bool(events),
    "deliveries": ["GATEWAY_APPLICATION_LOGS", "MEMORY_APPLICATION_LOGS"] + (["GATEWAY_TRACES"] if trace_delivery_ready else []),
    "span_contract": required_span_checks,
    "production_contract": production_contract,
    "redaction_checks": redaction_checks,
    "illustrative_pattern_spans": True,
})
print("Validation artifact written.")


## Takeaways

Strands supplied real agent, model, and tool spans around one Gateway call. AgentCore supplied managed Gateway metrics, logs, and traces plus real Memory event, retrieval, extraction, and consolidation signals. Illustrative custom spans showed the application fields to add to Graph, Workflow, Swarm-handoff, and A2A implementations.

A complete evidence chain uses a safe session identifier to connect application operations, managed service telemetry, and labelled evaluations. Keep sensitive prompt and tool content out of span attributes, propagate trace context across real boundaries, and fail a release when routing, evidence, quality, safety, or operational thresholds are not met.


## Clean up

Run the final cell even in a Workshop Studio account. It removes the synthetic Lab 6 Memory data, then uses the shared ownership record from Labs 1, 4, and 6 to remove only workshop-created delivery objects, log groups, the Logs resource policy, and Transaction Search settings. Existing shared configuration is left unchanged.

If a self-paced notebook deployed the Lab 1 or Lab 4 stack and kept it for this lab, `cleanup()` deletes that notebook-owned stack. Workshop Studio event stacks are detected and left in place.

To turn Transaction Search off later, open **CloudWatch settings → Transaction Search** and disable it, or call `xray.update_trace_segment_destination(Destination="XRay")`. The cell below restores the previous destination when the workshop enabled Transaction Search.

In [ ]:
def delete_owned(label, owned, function, **kwargs):
    if not owned:
        print(f"Leaving {label}: it existed before this notebook run.")
        return False
    try:
        function(**kwargs)
        print(f"Deleted {label}.")
    except ClientError as error:
        if is_missing(error):
            print(f"Already deleted {label}.")
        else:
            raise
    return True


# Remove the synthetic Memory data before deleting any notebook-owned Lab 4 stack.
# Refresh the unique Lab 6 namespace so a record that finished extraction late is also removed.
late_records = memory.list_memory_records(
    memoryId=memory_id,
    namespace=memory_namespace,
    maxResults=100,
).get("memoryRecordSummaries", [])
tracked_memory_records = {
    item["memoryRecordId"]: item
    for item in observability_cleanup.get("memory_records", [])
}
for item in late_records:
    tracked_memory_records[item["memoryRecordId"]] = {
        "memoryId": memory_id,
        "memoryRecordId": item["memoryRecordId"],
        "namespace": memory_namespace,
    }
observability_cleanup["memory_records"] = list(tracked_memory_records.values())

for record in list(observability_cleanup.get("memory_records", [])):
    delete_owned(
        f"memory record {record['memoryRecordId']}",
        True,
        memory_data.delete_memory_record,
        **record,
    )
observability_cleanup["memory_records"] = []

memory_event_cleanup = observability_cleanup.get("memory_event")
if memory_event_cleanup:
    delete_owned(
        f"memory event {memory_event_cleanup['eventId']}",
        True,
        memory_data.delete_event,
        **memory_event_cleanup,
    )
    observability_cleanup["memory_event"] = None

for label, delivery_id in list(observability_cleanup["deliveries"].items()):
    if delete_owned(f"{label} delivery", True, logs.delete_delivery, id=delivery_id):
        del observability_cleanup["deliveries"][label]

for name, owned in list(observability_cleanup["delivery_sources"].items()):
    if delete_owned(f"delivery source {name}", owned, logs.delete_delivery_source, name=name):
        observability_cleanup["delivery_sources"][name] = False

for name, owned in list(observability_cleanup["delivery_destinations"].items()):
    if delete_owned(
        f"delivery destination {name}", owned, logs.delete_delivery_destination, name=name
    ):
        observability_cleanup["delivery_destinations"][name] = False

for name, owned in list(observability_cleanup.get("log_groups", {}).items()):
    if delete_owned(f"log group {name}", owned, logs.delete_log_group, logGroupName=name):
        observability_cleanup["log_groups"][name] = False

if observability_cleanup.get("indexing_changed"):
    previous_indexing_percentage = observability_cleanup.get("previous_indexing_percentage")
    xray.update_indexing_rule(
        Name="Default",
        Rule={"Probabilistic": {"DesiredSamplingPercentage": previous_indexing_percentage}},
    )
    print(f"Restored trace indexing to {previous_indexing_percentage}%.")
    observability_cleanup["indexing_changed"] = False
    observability_cleanup["previous_indexing_percentage"] = None
else:
    print("Leaving trace indexing unchanged: it was already 100% before this notebook run.")

if observability_cleanup["transaction_search_changed"]:
    previous = observability_cleanup["previous_trace_destination"] or "XRay"
    current = xray.get_trace_segment_destination()
    if current.get("Status") == "PENDING":
        current = wait_for_trace_destination(
            current.get("Destination") or previous, timeout_seconds=60, required=False
        )
    if current.get("Destination") != previous:
        xray.update_trace_segment_destination(Destination=previous)
        current = wait_for_trace_destination(previous, timeout_seconds=60, required=False)
    transaction_search_restored = current.get("Status") != "PENDING"
    if transaction_search_restored:
        print(f"Restored trace segment destination to {previous}.")
    else:
        print(
            f"AWS accepted the restore to {previous} and is completing it in the background. "
            "Rerun this cleanup cell until it confirms the destination is restored."
        )
else:
    transaction_search_restored = True
    print("Leaving Transaction Search unchanged: it was enabled before this notebook run.")

if transaction_search_restored and observability_cleanup["resource_policy_created"]:
    if delete_owned(
        f"Logs resource policy {resource_policy_name}",
        True,
        logs.delete_resource_policy,
        policyName=resource_policy_name,
    ):
        observability_cleanup["resource_policy_created"] = False
elif transaction_search_restored and (
    observability_cleanup["transaction_search_changed"]
    and observability_cleanup["previous_resource_policy_document"] is not None
):
    logs.put_resource_policy(
        policyName=resource_policy_name,
        policyDocument=observability_cleanup["previous_resource_policy_document"],
    )
    observability_cleanup["previous_resource_policy_document"] = None
    print(f"Restored Logs resource policy {resource_policy_name}.")
else:
    print(f"Leaving Logs resource policy {resource_policy_name} unchanged.")

if transaction_search_restored:
    observability_cleanup["transaction_search_changed"] = False

cleanup("agents-as-tools", region=region)
cleanup("swarm", region=region)

if transaction_search_restored:
    delete_observability_state(observability_state_path)
    print("Removed the shared observability cleanup record.")
else:
    save_observability_state(observability_state_path, observability_cleanup)
    print("Saved cleanup progress. Rerun this cell after Transaction Search finishes restoring.")
